# 35 · Durable execution: checkpoints, Temporal and a worker-kill drill
Level L6 · Part 6 · core · ~30 min · builds on 19, 32

Needs Docker with Compose (one small container). Without Docker, the drill cells show the recorded run,
labelled as recorded output, and the rest of the lesson still runs.

## Goal
Kill the worker in the middle of a refund run, restart it, and check that the run finishes once and only once.

## Why it matters
Lesson 32 made one refund safe to retry, but the retry loop lived inside one process. If that process dies after the
payment has moved (a deploy, an out-of-memory kill, a lost node), the loop dies with it. The next attempt starts from the
top. The cell below runs a refund with no record outside the process, and the process dies after the charge.

In [1]:
charges = []  # the provider's ledger: every charge that moved money (no key here)


def charge(order: str, amount: float) -> None:
    charges.append((order, amount))  # the money moves here


def refund_run(order: str, amount: float, crash: bool) -> str:  # the whole run, in one process
    charge(order, amount)  # step 1: the money moves
    if crash:
        raise RuntimeError("the worker died before it recorded step 1")  # the process is gone
    return "receipt sent"  # step 2


try:
    refund_run("2207", 84.00, crash=True)
except RuntimeError as err:
    print("crash:", err)
refund_run("2207", 84.00, crash=False)  # the restart starts from the top: nothing outside the process says step 1 ran
print("charges:", len(charges), "· paid:", sum(a for _, a in charges), "USD for an 84.00 USD order")

crash: the worker died before it recorded step 1
charges: 2 · paid: 168.0 USD for an 84.00 USD order


This is the failure Temporal prevents; the real drill starts at Step 3.

**What it makes easy:** a run that resumes at the first step it has not finished, on any worker, after any crash.
**What it costs:** a record kept outside the process, a worker that reads it, and the rule that a step can run twice.
**When you don't need it:** a run that lasts seconds inside one process, where lesson 32's retries are enough.

## The idea
A durable run writes down each step it finishes, outside the process. After a crash, another worker reads that record
and continues at the first step with no record. A **checkpoint** (lesson 19's LangGraph saver) stores the state after
each node. **Event-history replay** (Temporal) stores each finished step's result: the workflow code runs again from its
first line, and finished steps return their saved results. Rerunning does not redo finished steps, because their saved
results are returned instead. Only the step with no record runs for real.

Analogy: a kitchen ticket with a tick per course, so the next cook starts at the first untick.

## Picture
```mermaid
sequenceDiagram
  participant A as Worker A
  participant S as Temporal server (event history)
  participant P as Payment provider
  participant B as Worker B
  S->>A: task: charge the refund
  A->>P: charge, key from the run id
  P-->>P: money moves, ledger row written
  Note over A: worker A is killed, no reply comes back
  S->>S: start-to-close timeout (3 s) expires
  S->>B: retry the charge (attempt 2), same key refund:run_id
  B->>P: same key
  P-->>B: saved answer, no second charge
  B-->>S: charged, then the receipt, run completes
```

## Step by step
**Step 1 · the provider and the worker (no Docker, no model).** This lesson ships its code as files, as lesson 34 did.
`payments.py` is a mock payment provider: a SQLite ledger that every worker process shares, where a repeated key returns
the saved answer. `worker.py` holds one workflow with two steps (the charge, then the receipt) and the worker that runs
them. The orders are fictional, as in lessons 06 to 14. The two shipped files are the drill's code, so read them as
you go. First, the folders and the settings the workers read.

In [2]:
import asyncio
import json
import os
import shutil
import sqlite3
import subprocess
import sys
import time
import uuid
from contextlib import closing
from pathlib import Path

from temporalio.client import Client

sys.path.insert(0, "..")  # the course folder: tut_env.py and llm_replay.py live there, so these imports find them
from tut_env import env  # course helper: reads a setting from the .env file (not part of the SDK)

Path("../apps/35/refunds").mkdir(parents=True, exist_ok=True)  # the code this lesson writes and runs
Path("../data/35").mkdir(parents=True, exist_ok=True)           # this lesson's run-time folder
os.environ["PAYMENTS_DB"] = str(Path("../data/35/payments.db").resolve())  # read by payments.py: the ledger, one file for every worker
os.environ["RECEIPTS_FILE"] = str(Path("../data/35/receipts.txt").resolve())  # read by worker.py: where receipts are appended
for name in ("payments.db", "receipts.txt"):  # a clean run: nothing left from an earlier try
    Path("../data/35", name).unlink(missing_ok=True)

The payment provider. It is a mock for this course. A charge is written to the ledger **before** the slow reply
(`PAY_REPLY_DELAY_S`), so a worker that dies while waiting has still moved the money.

In [3]:
%%writefile ../apps/35/refunds/payments.py
"""The payment provider of lesson 35: a mock for this course, not a real service.
Charges go to a SQLite ledger that every worker process shares. A repeated key returns the saved answer.
The orders are fictional, as in lessons 06 to 14."""
import os
import sqlite3
import time
from contextlib import closing
from pathlib import Path

ORDERS = {"1093": 45.00, "2207": 84.00}                  # order totals
DB = Path(os.environ.get("PAYMENTS_DB", "payments.db"))  # set by the notebook: one file for all workers


class Refused(Exception):
    """A permanent answer: the amount is over the order total. A retry gets the same answer."""


def connect() -> sqlite3.Connection:
    conn = sqlite3.connect(DB, timeout=10)
    conn.execute("CREATE TABLE IF NOT EXISTS charges (key TEXT PRIMARY KEY, order_id TEXT, amount REAL, answer TEXT)")
    conn.execute("CREATE TABLE IF NOT EXISTS calls (key TEXT, attempt INTEGER, outcome TEXT)")
    return conn


def charge(order_id: str, amount_usd: float, key: str, attempt: int = 1) -> str:
    with closing(connect()) as conn, conn:  # one transaction: committed when the block ends
        row = conn.execute("SELECT answer FROM charges WHERE key = ?", (key,)).fetchone()
        if row:  # the same key: the saved answer, and no second charge
            conn.execute("INSERT INTO calls VALUES (?, ?, 'saved answer')", (key, attempt))
            return row[0]
        if amount_usd > ORDERS[order_id]:
            raise Refused(f"{amount_usd:.2f} USD is over the order total")
        n = conn.execute("SELECT COUNT(*) FROM charges").fetchone()[0] + 1
        conn.execute("INSERT INTO charges VALUES (?, ?, ?, ?)", (key, order_id, amount_usd, f"charge #{n}"))
        conn.execute("INSERT INTO calls VALUES (?, ?, 'charged')", (key, attempt))
    # the money has moved and is committed; the reply is slow here, and a crash now loses only the reply
    time.sleep(float(os.environ.get("PAY_REPLY_DELAY_S", "0")))
    return f"charge #{n}"

Overwriting ../apps/35/refunds/payments.py


The worker. The workflow is the run's plan: it calls the charge, then the receipt. Each call is an **activity**, and the
server records each activity's result. A charge that gets no reply is tried again after 3 seconds (the start-to-close
timeout), on whichever worker is listening. The key comes from the run id, so every try sends the same key.

In [4]:
%%writefile ../apps/35/refunds/worker.py
"""The refund run of lesson 35: one workflow, two activities and the worker that runs them.
The notebook starts this file as its own process, so the drill can kill it. The run's record lives in the
Temporal server, not in this process."""
import asyncio
import os
from datetime import timedelta

from temporalio import activity, workflow
from temporalio.client import Client
from temporalio.common import RetryPolicy
from temporalio.exceptions import ApplicationError
from temporalio.worker import Worker

with workflow.unsafe.imports_passed_through():
    import payments  # the mock payment provider, next to this file

QUEUE = "refunds-35"  # the task queue every worker listens on


@activity.defn
async def charge_card(refund: dict) -> str:
    info = activity.info()
    key = f"refund:{info.workflow_id}"  # made from the run, not the attempt: every retry sends the same key
    try:  # to_thread: a slow reply waits in a thread, not in the worker's event loop
        return await asyncio.to_thread(payments.charge, refund["order_id"], refund["amount_usd"], key, info.attempt)
    except payments.Refused as err:
        raise ApplicationError(str(err), type="Refused", non_retryable=True) from err  # one try, then the run fails


@activity.defn
async def send_receipt(refund: dict) -> str:
    with open(os.environ["RECEIPTS_FILE"], "a") as f:  # one line per receipt sent
        f.write(f"receipt for order {refund['order_id']}\n")
    return "receipt sent"


@workflow.defn
class RefundRun:
    @workflow.run
    async def run(self, refund: dict) -> dict:
        charge = await workflow.execute_activity(
            "charge_card", refund, start_to_close_timeout=timedelta(seconds=3),  # a lost worker is noticed after 3 s
            retry_policy=RetryPolicy(initial_interval=timedelta(seconds=0.2), backoff_coefficient=2.0, maximum_attempts=6))
        receipt = await workflow.execute_activity("send_receipt", refund, start_to_close_timeout=timedelta(seconds=3))
        return {"charge": charge, "receipt": receipt}


async def main() -> None:
    client = await Client.connect(os.environ["TEMPORAL_ADDRESS"])
    worker = Worker(client, task_queue=QUEUE, workflows=[RefundRun], activities=[charge_card, send_receipt])
    print("worker ready", flush=True)
    await worker.run()


if __name__ == "__main__":
    asyncio.run(main())

Overwriting ../apps/35/refunds/worker.py


**Step 2 · the agent asks for the refund (needs-llm).** In lesson 32 the support agent asks for the refund through one
tool, `issue_refund`. That is the only model call of this notebook. The durable run below pays what the model asked for.
The model is called through the official Google SDK. `llm_replay` saves the call once, and a recording replays it with
no key.

In [5]:
import llm_replay  # course helper: records and replays model calls (not part of the SDK)
llm_replay.start("35_durable-execution")

from google import genai  # the official Google SDK
from google.genai import types

MODEL = env("TUT_LLM_MODEL", "gemini-3.1-flash-lite")  # the course's default model
client = genai.Client(
    vertexai=True, api_key="replay" if llm_replay.is_replaying() else env("GOOGLE_CLOUD_API_KEY"),
    http_options=types.HttpOptions(timeout=30_000, retry_options=types.HttpRetryOptions(
        attempts=4, initial_delay=5.0, max_delay=30.0)))  # the SDK's own retry for 429 and 5xx
ISSUE = types.FunctionDeclaration(
    name="issue_refund", description="Pay an approved refund for one order.",
    parameters_json_schema={"type": "object", "properties": {"order_id": {"type": "string"},
                            "amount_usd": {"type": "number"}}, "required": ["order_id", "amount_usd"]})
config = types.GenerateContentConfig(
    system_instruction="You are the support assistant for Brightside Outfitters. The refund is approved: call issue_refund.",
    tools=[types.Tool(function_declarations=[ISSUE])], temperature=0)
reply = client.models.generate_content(  # the only model call of this notebook
    model=MODEL, contents="Order 2207 arrived with a torn box. The refund of 84.00 USD is approved. Please issue it.",
    config=config)
llm_replay.stop()
call = next(p.function_call for p in reply.candidates[0].content.parts if p.function_call)
args = dict(call.args)
print("model asked:", call.name, args)

LLM mode: replay (35_durable-execution.yaml)


model asked: issue_refund {'amount_usd': 84, 'order_id': '2207'}


**Fallback (only if the model cell was skipped).** The same arguments are typed in by hand, as the model would send
them. It is not model output, and the rest of the notebook still runs. The model's arguments are data: the code converts
them and checks the order against the provider's list before any money moves.

In [6]:
sys.path.insert(0, "../apps/35/refunds")  # the mock provider and the worker, written above
import payments

args = globals().get("args", {"order_id": "2207", "amount_usd": 84.0})  # the fallback when the model cell was skipped
REFUND = {"order_id": str(args["order_id"]), "amount_usd": float(args["amount_usd"])}
assert 0 < REFUND["amount_usd"] <= payments.ORDERS.get(REFUND["order_id"], 0), "refusing a refund the desk cannot make"
print("refund to run:", REFUND, "· orders the mock provider knows:", payments.ORDERS)

refund to run: {'order_id': '2207', 'amount_usd': 84.0} · orders the mock provider knows: {'1093': 45.0, '2207': 84.0}


**Step 3 · the Temporal server (needs-docker).** The Temporal dev server runs in Compose (`infra/durable/compose.yml`,
one pinned image). It keeps each run's event history, so a worker that dies does not take the run with it. The check
cell says whether Docker is here. The start cell brings the server up and waits until a client can connect. Open the
web UI at `http://localhost:8351` while the drill runs.

In [7]:
HAS_DOCKER = bool(shutil.which("docker")) and subprocess.run(
    ["docker", "compose", "version"], capture_output=True).returncode == 0
ADDRESS = env("TEMPORAL_ADDRESS", "") or "localhost:8350"  # the Compose dev server's gRPC port
COMPOSE = ["docker", "compose", "-p", "tut-agentic-ai-15h-35", "-f", "../infra/durable/compose.yml"]
QUEUE = "refunds-35"  # the same task queue as worker.py
RECORDED_FILE = Path("../data/recorded/35.json")  # the real run, shipped with the course
RECORDED = json.loads(RECORDED_FILE.read_text()) if RECORDED_FILE.exists() else {}
RESULTS = {}  # what the Docker cells measured here
print("docker with compose:", HAS_DOCKER, "· recorded results shipped for:", sorted(RECORDED))

docker with compose: True · recorded results shipped for: ['calls', 'charges', 'events', 'money_moved_before_kill', 'receipts', 'result', 'server', 'worker_a_ready']


**Start the server (needs-docker).** Compose starts the container, and then the poll connects a client, the same way
the workers do. The cell returns only when the server answers. The first start pulls the image, so it is slow.

In [8]:
# top-level await works only in notebooks; use asyncio.run in a script
async def server_ready(timeout_s: float = 60) -> bool:  # poll until a client connects (no fixed sleep)
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        try:
            await Client.connect(ADDRESS)
            return True
        except Exception:
            await asyncio.sleep(1)
    return False


if HAS_DOCKER:
    subprocess.run(COMPOSE + ["up", "-d", "--wait"], check=True, capture_output=True, text=True)
    RESULTS["server"] = await server_ready()
    print("temporal server reachable:", RESULTS["server"])

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


Helpers for the drill: start a worker as its own process, and count the ledger rows. A worker's log goes to
`../data/35/`. Worker A's charge reply is 5 seconds late, so the drill has time to kill it.

In [9]:
LOG_DIR, WORKER = Path("../data/35"), Path("../apps/35/refunds/worker.py").resolve()


def wait_until(check, timeout_s: float = 60) -> bool:  # poll a condition, with a timeout
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        if check():
            return True
        time.sleep(0.2)
    return False


def start_worker(name: str, reply_delay_s: float) -> subprocess.Popen:
    log = open(LOG_DIR / f"{name}.log", "w")
    env_ = {**os.environ, "TEMPORAL_ADDRESS": ADDRESS, "PAY_REPLY_DELAY_S": str(reply_delay_s)}
    proc = subprocess.Popen([sys.executable, "-u", str(WORKER)], cwd=WORKER.parent, env=env_,
                            stdout=log, stderr=subprocess.STDOUT)
    wait_until(lambda: "worker ready" in (LOG_DIR / f"{name}.log").read_text())  # returns once the worker logs ready
    return proc


def charges_in_ledger() -> int:
    with closing(payments.connect()) as conn:
        return conn.execute("SELECT COUNT(*) FROM charges").fetchone()[0]

**Step 4 · the drill (needs-docker).** Start worker A, and start the refund. Wait until the ledger shows the charge:
the money has moved, and the reply is still 5 seconds away. Then kill worker A with SIGKILL, which gives it no time to
clean up. Start worker B with a normal reply. Worker B waits for the server to time the attempt out, then runs it again.

In [10]:
RUN_ID = "refund-2207-1"  # the workflow id: the run's name, and the root of its payment key
if HAS_DOCKER and RESULTS.get("server"):
    worker_a = start_worker("worker-a", reply_delay_s=5)
    client = await Client.connect(ADDRESS)
    handle = await client.start_workflow("RefundRun", REFUND, id=RUN_ID, task_queue=QUEUE)
    RESULTS["money_moved_before_kill"] = wait_until(lambda: charges_in_ledger() == 1)
    print("worker A ready and the charge in the ledger:", RESULTS["money_moved_before_kill"])

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


**Kill worker A, finish on worker B (needs-docker).** The kill is the failure. The recovery is worker B connecting, the
server timing out attempt 1, and the run completing. The notebook does not call the charge itself.

In [11]:
if HAS_DOCKER and RESULTS.get("money_moved_before_kill"):
    worker_a.kill()  # SIGKILL: no shutdown, no cleanup, and the charge's reply never arrives
    worker_a.wait()
    print("worker A killed · charges in the ledger:", charges_in_ledger())
    worker_b = start_worker("worker-b", reply_delay_s=0)  # the restarted worker, with a normal reply
    RESULTS["result"] = await asyncio.wait_for(handle.result(), timeout=90)  # the run finishes on worker B
    print("run result:", RESULTS["result"])

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


**Gather the checks (needs-docker).** The ledger, the receipts file and the server's history are read after the run,
not from the workers' logs.

In [12]:
from temporalio.api.enums.v1 import EventType

if HAS_DOCKER and "result" in RESULTS:  # the checks: what the ledger, the log and the event history say
    with sqlite3.connect(os.environ["PAYMENTS_DB"]) as conn:
        RESULTS["calls"] = conn.execute("SELECT attempt, outcome FROM calls ORDER BY rowid").fetchall()
        RESULTS["charges"] = conn.execute("SELECT answer FROM charges").fetchall()
    RESULTS["receipts"] = Path(os.environ["RECEIPTS_FILE"]).read_text().count("\n")
    events = [EventType.Name(e.event_type).removeprefix("EVENT_TYPE_") async for e in handle.fetch_history_events()]
    RESULTS["events"] = [e for e in events if e.startswith(("ACTIVITY_TASK", "WORKFLOW_EXECUTION"))]

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


**Result.** Live when Docker ran here. Otherwise each line shows the recorded output of the real run, labelled as such.
The check: `charges` has 1 row, `receipts` is 1, and `calls` is `[(1, 'charged'), (2, 'saved answer')]`. Attempt 1
charged, and attempt 2 found the saved answer, so the ledger holds one charge. The `events` line is the server's
history: one scheduled, started and completed event per activity. The timed-out first try is not a separate event.
Temporal keeps retries in the activity's state and writes the final result.

In [13]:
for key in ("money_moved_before_kill", "result", "calls", "charges", "receipts", "events"):  # live if Docker ran, else recorded
    print(f"{key}:", RESULTS.get(key, "recorded output (the drill did not run here): " + str(RECORDED.get(key))))

money_moved_before_kill: recorded output (the drill did not run here): True
result: recorded output (the drill did not run here): {'charge': 'charge #1', 'receipt': 'receipt sent'}
calls: recorded output (the drill did not run here): [[1, 'charged'], [2, 'saved answer']]
charges: recorded output (the drill did not run here): [['charge #1']]
receipts: recorded output (the drill did not run here): 1
events: recorded output (the drill did not run here): ['WORKFLOW_EXECUTION_STARTED', 'ACTIVITY_TASK_SCHEDULED', 'ACTIVITY_TASK_STARTED', 'ACTIVITY_TASK_COMPLETED', 'ACTIVITY_TASK_SCHEDULED', 'ACTIVITY_TASK_STARTED', 'ACTIVITY_TASK_COMPLETED', 'WORKFLOW_EXECUTION_COMPLETED']


**Step 5 · tear down (needs-docker).** The workers stop, the Compose project goes, and its volumes with it. The
dev server keeps its history in memory, so the runs are gone. The results are saved to `../data/35/run.json`.

In [14]:
for proc in (globals().get("worker_a"), globals().get("worker_b")):
    if proc is not None and proc.poll() is None:
        proc.kill()
        proc.wait()
if HAS_DOCKER:
    subprocess.run(COMPOSE + ["down", "-v", "--remove-orphans"], capture_output=True, text=True)
    print("compose down: server and history removed")
_ = Path("../data/35/run.json").write_text(json.dumps(RESULTS, indent=2, default=str))

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


## At real scale
- **The record is the state.** Temporal keeps the event history on the server. Any worker on the task queue can continue
  the run, so a deploy or a crash is a restart, not a lost refund. Where that record lives when a region fails is
  lesson 36 (HA/DR).
- **A dead worker is found by a timeout.** The server cannot see a crashed worker, so the start-to-close timeout is the
  detector, and it forces the retry ([Temporal docs](https://docs.temporal.io/encyclopedia/detecting-activity-failures)).
  Set it above the slowest honest reply. A shorter one starts a second try while the first still runs, so check how the
  provider answers a repeat that arrives during the first call.

## Decision guide
| Situation | Approach | Why |
|---|---|---|
| A call that fails and is retried within seconds (lesson 32) | retry with a timeout and one key | nothing outlives the process |
| A run that pauses for a person, in one service (lesson 19) | a LangGraph checkpointer on a database file | state is saved after each node |
| A run that lasts hours, crosses deploys, or must survive a worker crash | a durable workflow engine (Temporal) | the server keeps the record |
| Any write to money or a customer | an idempotency key made from the run id, always | a step can run twice after a crash |

## What a durable run does not solve
- **Histories grow.** Temporal warns at 10,240 events and terminates a run above 51,200. History is never truncated
  ([Temporal docs](https://docs.temporal.io/workflow-execution/event)). A long loop closes and starts a new run
  (Continue-As-New). Go deeper: Continue-As-New and worker versioning, which keep runs in flight safe when code changes.
- **The dev server is for the lab.** `start-dev` keeps history in memory. Production runs the server against a database,
  with several server processes. The Compose file is the drill's stand-in, not a cluster.
- **Next:** the capstone (lesson 37) runs this drill on the whole support desk.

## Common mistakes
1. **A new key on every try.** A random key per attempt looks safe, and it charges twice. The key must come from the
   run, not the attempt. The cell below uses its own ledger file, so it does not touch the drill's.

In [15]:
payments.DB = Path("../data/35/mistake.db")  # a separate ledger for this cell only
payments.DB.unlink(missing_ok=True)
for attempt in (1, 2):  # the retry after a lost reply
    payments.charge("2207", 84.00, key=f"refund:{uuid.uuid4()}", attempt=attempt)  # WRONG: a new key on each try
with closing(payments.connect()) as conn:
    n = conn.execute("SELECT COUNT(*) FROM charges").fetchone()[0]
assert n == 1, f"double charge: {n} charges for one refund"

AssertionError: double charge: 2 charges for one refund

The assert stops the cell with an `AssertionError`: the provider charged twice. The fix is a key made from the run id,
as `charge_card` builds it, so both tries send the same key.

In [16]:
payments.DB.unlink(missing_ok=True)  # the same separate ledger, emptied
RUN = "refund-2207-fixed"  # made once per run, like the workflow id
for attempt in (1, 2):
    payments.charge("2207", 84.00, key=f"refund:{RUN}", attempt=attempt)  # RIGHT: the key comes from the run
with closing(payments.connect()) as conn:
    print("charges for one 84.00 USD refund:", conn.execute("SELECT COUNT(*) FROM charges").fetchone()[0])

charges for one 84.00 USD refund: 1


## Try it
Write `key_for(run_id, attempt)`. It must return one key for every attempt of one run, and a different key for another
run. Use the same format as `charge_card`. Check that the attempts 1 and 2 of run `refund-2207-1` share a key, and that
run `refund-2207-2` gets a different one.

In [17]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [18]:
def key_for(run_id: str, attempt: int) -> str:
    return f"refund:{run_id}"  # the attempt is not part of the key: every retry sends the same one


print(key_for("refund-2207-1", 1) == key_for("refund-2207-1", 2), key_for("refund-2207-1", 1) != key_for("refund-2207-2", 1))

True True


## Recap
- A durable run keeps finished steps outside the process, so any worker resumes at the first unfinished one.
- A checkpoint saves state (lesson 19). Temporal saves each step's result and replays the code.
- A step can act and lose its record, so it runs again: the key from the run id makes the repeat safe.
- The drill: worker A died during a charge; worker B got the saved answer. One charge, one receipt.

**Interview:** "The worker dies after the payment call but before it saves the result. How do you stop a double charge?"
Strong answer: "Retry on another worker with the same idempotency key, made from the run id. The provider returns the
saved answer, so the money moves once."